<a href="https://colab.research.google.com/github/yazankhwork/secure-medical-record-exchange/blob/main/Secure_Medical_Record_Exchange_GitHub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Secure Medical Record Exchange

This notebook demonstrates an educational end-to-end system for securely
transmitting a medical record over an untrusted channel.

The system combines:

- **IDEA in CFB mode** for medical-record encryption
- **Kyber/ML-KEM-style key encapsulation** for shared-key delivery
- **Falcon-style NTRU signatures** for authenticity and integrity

## Security flow

1. The doctor generates a public/private KEM key pair.
2. The patient encapsulates a shared secret using the doctor's public key.
3. An IDEA key is derived from the shared secret.
4. The medical record is encrypted with IDEA-CFB.
5. The complete encrypted package is signed.
6. The doctor verifies the signature before processing the package.
7. The doctor recovers the shared secret and decrypts the record.
8. Tampering tests demonstrate signature rejection.

> This project is an educational implementation and is not intended for
> production cryptographic or medical use. All patient data are synthetic.

In [ ]:
# @title Cell 1 — Input Data -Memory Medical Record Input
# This version does not create folders or write Python files.
# The medical record is stored directly in Colab memory.

MEDICAL_RECORD_TEXT = r'''
==================================================================
            CONFIDENTIAL MEDICAL RECORD
==================================================================
Patient Name : Jane A. Doe
Patient ID   : PT-2026-00471
Date of Birth: 1989-03-14
Sex          : Female
Blood Type   : O+

------------------------------------------------------------------
VISIT SUMMARY
------------------------------------------------------------------
Date of Visit : 2026-06-10
Attending     : Dr. M. Halpern, Internal Medicine
Chief Complaint: Recurrent headaches and elevated blood pressure.

Vitals:
  - Blood Pressure : 148 / 95 mmHg
  - Heart Rate     : 82 bpm
  - Temperature    : 36.8 C
  - Weight         : 71 kg

------------------------------------------------------------------
DIAGNOSIS
------------------------------------------------------------------
  - Stage 1 Hypertension (ICD-10: I10)
  - Tension-type headache (ICD-10: G44.2)

------------------------------------------------------------------
MEDICATIONS PRESCRIBED
------------------------------------------------------------------
  - Lisinopril 10 mg, once daily
  - Ibuprofen 400 mg, as needed for headache (max 3x/day)

------------------------------------------------------------------
LAB RESULTS
------------------------------------------------------------------
  - Fasting glucose : 92 mg/dL  (normal)
  - LDL cholesterol : 138 mg/dL (borderline high)
  - Creatinine      : 0.9 mg/dL (normal)

------------------------------------------------------------------
NOTES
------------------------------------------------------------------
Patient advised to reduce sodium intake, begin light daily
exercise, and return in 4 weeks for a blood-pressure recheck.
Referral to cardiology if BP remains above 140/90.

== END OF RECORD ==

'''.strip() + "\n"

MEDICAL_RECORD_BYTES = MEDICAL_RECORD_TEXT.encode("utf-8")
MEDICAL_RECORD_SOURCE_DESCRIPTION = "Colab memory variable: MEDICAL_RECORD_TEXT (no external file)"


In [ ]:
# @title Cell 2 — Shared Utility Functions
"""
utils.py
--------
Shared utility functions used by the cryptographic modules.

These helpers wrap Python standard-library hash functions and provide the
byte/bit conversions required by the Kyber implementation.
"""

import hashlib


# ---------------------------------------------------------------------------
# Hash and extendable-output functions.
# ---------------------------------------------------------------------------

def sha3_256(input_data: bytes) -> bytes:
    """Return the 32-byte SHA3-256 digest of input_data."""
    return hashlib.sha3_256(input_data).digest()


def sha3_512(input_data: bytes) -> bytes:
    """Return the 64-byte SHA3-512 digest of input_data."""
    return hashlib.sha3_512(input_data).digest()


def shake_128(input_data: bytes, output_length: int) -> bytes:
    """Return output_length bytes produced by SHAKE-128."""
    return hashlib.shake_128(input_data).digest(output_length)


def shake_256(input_data: bytes, output_length: int) -> bytes:
    """Return output_length bytes produced by SHAKE-256."""
    return hashlib.shake_256(input_data).digest(output_length)


# ---------------------------------------------------------------------------
# Byte-to-bit and bit-to-byte conversion helpers.
# Bits are ordered least-significant-bit first inside each byte, as in Kyber.
# ---------------------------------------------------------------------------

def bytes_to_bits(input_bytes: bytes) -> list:
    """Convert bytes into a flat list of bits, least-significant bit first."""
    bit_values = []
    for current_byte in input_bytes:
        for bit_position in range(8):
            bit_values.append((current_byte >> bit_position) & 1)
    return bit_values


def bits_to_bytes(bit_values: list) -> bytes:
    """Convert a bit list back to bytes. Its length must be divisible by 8."""
    output_bytes = bytearray(len(bit_values) // 8)
    for bit_index, bit_value in enumerate(bit_values):
        byte_index = bit_index // 8
        bit_position = bit_index % 8
        output_bytes[byte_index] |= (bit_value & 1) << bit_position
    return bytes(output_bytes)


def xor_bytes(first_bytes: bytes, second_bytes: bytes) -> bytes:
    """Return the byte-wise XOR of two equal-length byte strings."""
    return bytes(
        first_byte ^ second_byte
        for first_byte, second_byte in zip(first_bytes, second_bytes)
    )


# A small namespace object keeps the same style used by the project modules,
# without creating any local Python files.
from types import SimpleNamespace

utils = SimpleNamespace(
    sha3_256=sha3_256,
    sha3_512=sha3_512,
    shake_128=shake_128,
    shake_256=shake_256,
    bytes_to_bits=bytes_to_bits,
    bits_to_bytes=bits_to_bytes,
    xor_bytes=xor_bytes,
)


In [ ]:
# @title Cell 3 — IDEA-CFB Symmetric Encryption
from types import SimpleNamespace

def build_idea_cfb_module():
    """
    idea_cfb.py
    -----------
    A pure-Python implementation of the IDEA block cipher in CFB mode.

    IDEA uses:
      * a 64-bit block split into four 16-bit words;
      * a 128-bit encryption key;
      * eight full rounds and one output transformation;
      * XOR, addition modulo 2**16, and multiplication modulo 2**16 + 1.

    CFB mode converts the block cipher into a stream-like cipher. Both encryption
    and decryption call the IDEA encryption block function and XOR its output with
    the data, so no padding is required.
    """

    import os

    ADDITION_MODULUS = 0x10000
    MULTIPLICATION_MODULUS = 0x10001
    WORD_MASK_16 = 0xFFFF
    BLOCK_SIZE_BYTES = 8
    KEY_SIZE_BYTES = 16
    TOTAL_SUBKEYS = 52


    # ---------------------------------------------------------------------------
    # IDEA operations on 16-bit words.
    # ---------------------------------------------------------------------------

    def _add_words(first_word: int, second_word: int) -> int:
        """Add two words modulo 2**16."""
        return (first_word + second_word) & WORD_MASK_16


    def _multiply_words(first_word: int, second_word: int) -> int:
        """Multiply modulo 2**16 + 1, where zero represents the value 65536."""
        if first_word == 0:
            first_word = ADDITION_MODULUS
        if second_word == 0:
            second_word = ADDITION_MODULUS

        multiplication_result = (
            first_word * second_word
        ) % MULTIPLICATION_MODULUS

        # The modular value 65536 is encoded as the 16-bit value zero.
        return multiplication_result & WORD_MASK_16


    def _multiplicative_inverse(word_value: int) -> int:
        """Return the IDEA multiplicative inverse modulo 2**16 + 1.

        This helper is retained for completeness. CFB mode only requires the IDEA
        encryption path, so it is not used by this project flow.
        """
        if word_value <= 1:
            return word_value
        return pow(
            word_value,
            MULTIPLICATION_MODULUS - 2,
            MULTIPLICATION_MODULUS,
        ) & WORD_MASK_16


    # ---------------------------------------------------------------------------
    # Expand the 128-bit IDEA key into 52 sixteen-bit subkeys.
    # ---------------------------------------------------------------------------

    def _expand_encryption_key(encryption_key: bytes) -> list:
        assert len(encryption_key) == KEY_SIZE_BYTES, (
            "IDEA key must be 128 bits (16 bytes)"
        )

        key_as_integer = int.from_bytes(encryption_key, "big")
        encryption_subkeys = []

        while len(encryption_subkeys) < TOTAL_SUBKEYS:
            for word_index in range(8):
                shift_amount = 16 * (7 - word_index)
                encryption_subkeys.append(
                    (key_as_integer >> shift_amount) & WORD_MASK_16
                )

            # Rotate the complete 128-bit key left by 25 bits.
            key_as_integer = (
                (key_as_integer << 25)
                | (key_as_integer >> (128 - 25))
            ) & ((1 << 128) - 1)

        return encryption_subkeys[:TOTAL_SUBKEYS]


    # ---------------------------------------------------------------------------
    # Encrypt one 64-bit block using IDEA.
    # ---------------------------------------------------------------------------

    def _encrypt_block(block_bytes: bytes, encryption_subkeys: list) -> bytes:
        first_word = int.from_bytes(block_bytes[0:2], "big")
        second_word = int.from_bytes(block_bytes[2:4], "big")
        third_word = int.from_bytes(block_bytes[4:6], "big")
        fourth_word = int.from_bytes(block_bytes[6:8], "big")

        for round_index in range(8):
            subkey_start_index = round_index * 6
            round_subkeys = encryption_subkeys[
                subkey_start_index:subkey_start_index + 6
            ]

            multiplied_first_word = _multiply_words(first_word, round_subkeys[0])
            added_second_word = _add_words(second_word, round_subkeys[1])
            added_third_word = _add_words(third_word, round_subkeys[2])
            multiplied_fourth_word = _multiply_words(fourth_word, round_subkeys[3])

            first_ma_product = _multiply_words(
                round_subkeys[4],
                multiplied_first_word ^ added_third_word,
            )
            second_ma_product = _multiply_words(
                round_subkeys[5],
                _add_words(
                    first_ma_product,
                    added_second_word ^ multiplied_fourth_word,
                ),
            )
            ma_sum = _add_words(first_ma_product, second_ma_product)

            first_word = multiplied_first_word ^ second_ma_product
            second_word = added_second_word ^ ma_sum
            third_word = added_third_word ^ second_ma_product
            fourth_word = multiplied_fourth_word ^ ma_sum

        output_first_word = _multiply_words(first_word, encryption_subkeys[48])
        output_second_word = _add_words(third_word, encryption_subkeys[49])
        output_third_word = _add_words(second_word, encryption_subkeys[50])
        output_fourth_word = _multiply_words(fourth_word, encryption_subkeys[51])

        return (
            output_first_word.to_bytes(2, "big")
            + output_second_word.to_bytes(2, "big")
            + output_third_word.to_bytes(2, "big")
            + output_fourth_word.to_bytes(2, "big")
        )


    # ---------------------------------------------------------------------------
    # IDEA-CFB encryption and decryption.
    # ---------------------------------------------------------------------------

    def encrypt(
        encryption_key: bytes,
        initialization_vector: bytes,
        plaintext_bytes: bytes,
    ) -> bytes:
        encryption_subkeys = _expand_encryption_key(encryption_key)
        assert len(initialization_vector) == BLOCK_SIZE_BYTES, (
            "IDEA/CFB IV must be 64 bits (8 bytes)"
        )

        encrypted_output = bytearray()
        feedback_block = initialization_vector

        for byte_offset in range(0, len(plaintext_bytes), BLOCK_SIZE_BYTES):
            plaintext_block = plaintext_bytes[
                byte_offset:byte_offset + BLOCK_SIZE_BYTES
            ]
            keystream_block = _encrypt_block(feedback_block, encryption_subkeys)
            ciphertext_block = bytes(
                plaintext_byte ^ keystream_block[byte_index]
                for byte_index, plaintext_byte in enumerate(plaintext_block)
            )
            encrypted_output += ciphertext_block

            if len(ciphertext_block) == BLOCK_SIZE_BYTES:
                feedback_block = ciphertext_block
            else:
                feedback_block = (
                    ciphertext_block
                    + keystream_block[len(ciphertext_block):]
                )

        return bytes(encrypted_output)


    def decrypt(
        encryption_key: bytes,
        initialization_vector: bytes,
        ciphertext_bytes: bytes,
    ) -> bytes:
        encryption_subkeys = _expand_encryption_key(encryption_key)
        assert len(initialization_vector) == BLOCK_SIZE_BYTES, (
            "IDEA/CFB IV must be 64 bits (8 bytes)"
        )

        decrypted_output = bytearray()
        feedback_block = initialization_vector

        for byte_offset in range(0, len(ciphertext_bytes), BLOCK_SIZE_BYTES):
            ciphertext_block = ciphertext_bytes[
                byte_offset:byte_offset + BLOCK_SIZE_BYTES
            ]
            keystream_block = _encrypt_block(feedback_block, encryption_subkeys)
            plaintext_block = bytes(
                ciphertext_byte ^ keystream_block[byte_index]
                for byte_index, ciphertext_byte in enumerate(ciphertext_block)
            )
            decrypted_output += plaintext_block

            if len(ciphertext_block) == BLOCK_SIZE_BYTES:
                feedback_block = ciphertext_block
            else:
                feedback_block = (
                    ciphertext_block
                    + keystream_block[len(ciphertext_block):]
                )

        return bytes(decrypted_output)


    def random_iv() -> bytes:
        """Generate a fresh random 64-bit initialization vector."""
        return os.urandom(BLOCK_SIZE_BYTES)


    return SimpleNamespace(
        encrypt=encrypt,
        decrypt=decrypt,
        random_iv=random_iv,
        BLOCK_SIZE_BYTES=BLOCK_SIZE_BYTES,
        KEY_SIZE_BYTES=KEY_SIZE_BYTES
    )

idea_cfb = build_idea_cfb_module()


In [ ]:
# @title Cell 4 — Educational Kyber/ML-KEM-Style Key Delivery
from types import SimpleNamespace

def build_kyber_module(utility_module):
    utils = utility_module
    """
    kyber.py
    --------
    An educational, from-scratch implementation of ML-KEM (CRYSTALS-Kyber)
    using the Kyber-512 parameter set.

    The implementation keeps the original project logic:
      * polynomial arithmetic is performed in Z_q[x] / (x^256 + 1);
      * schoolbook multiplication is used instead of the NTT;
      * coefficients are serialized using two bytes each;
      * the Fujisaki-Okamoto transform provides implicit rejection.

    This code is intended for learning and demonstration, not production use.
    """

    import os


    # Kyber-512 parameters.
    POLYNOMIAL_DEGREE = 256
    MODULUS = 3329
    MODULE_RANK = 2
    KEY_GENERATION_NOISE_PARAMETER = 3
    ENCRYPTION_NOISE_PARAMETER = 2
    VECTOR_COMPRESSION_BITS = 10
    SCALAR_COMPRESSION_BITS = 4


    # ---------------------------------------------------------------------------
    # Polynomial arithmetic in Z_q[x] / (x^256 + 1).
    # ---------------------------------------------------------------------------

    def poly_add(first_polynomial, second_polynomial):
        return [
            (first_coefficient + second_coefficient) % MODULUS
            for first_coefficient, second_coefficient
            in zip(first_polynomial, second_polynomial)
        ]


    def poly_sub(first_polynomial, second_polynomial):
        return [
            (first_coefficient - second_coefficient) % MODULUS
            for first_coefficient, second_coefficient
            in zip(first_polynomial, second_polynomial)
        ]


    def poly_mul(first_polynomial, second_polynomial):
        """Multiply and reduce modulo x^POLYNOMIAL_DEGREE + 1 and MODULUS."""
        unreduced_product = [0] * (2 * POLYNOMIAL_DEGREE)

        for first_degree in range(POLYNOMIAL_DEGREE):
            first_coefficient = first_polynomial[first_degree]
            if first_coefficient == 0:
                continue

            for second_degree in range(POLYNOMIAL_DEGREE):
                unreduced_product[first_degree + second_degree] += (
                    first_coefficient * second_polynomial[second_degree]
                )

        reduced_product = [0] * POLYNOMIAL_DEGREE
        for coefficient_index in range(POLYNOMIAL_DEGREE):
            reduced_product[coefficient_index] = (
                unreduced_product[coefficient_index]
                - unreduced_product[coefficient_index + POLYNOMIAL_DEGREE]
            ) % MODULUS

        return reduced_product


    # ---------------------------------------------------------------------------
    # Sampling helpers.
    # ---------------------------------------------------------------------------

    def _parse_uniform(expansion_seed: bytes) -> list:
        """Create a uniformly distributed polynomial from a seed."""
        pseudorandom_stream = utils.shake_128(
            expansion_seed,
            3 * POLYNOMIAL_DEGREE * 3,
        )
        sampled_coefficients = []
        stream_index = 0

        while len(sampled_coefficients) < POLYNOMIAL_DEGREE:
            first_candidate = (
                pseudorandom_stream[stream_index]
                | ((pseudorandom_stream[stream_index + 1] & 0x0F) << 8)
            )
            second_candidate = (
                (pseudorandom_stream[stream_index + 1] >> 4)
                | (pseudorandom_stream[stream_index + 2] << 4)
            )
            stream_index += 3

            if first_candidate < MODULUS:
                sampled_coefficients.append(first_candidate)

            if (
                len(sampled_coefficients) < POLYNOMIAL_DEGREE
                and second_candidate < MODULUS
            ):
                sampled_coefficients.append(second_candidate)

        return sampled_coefficients


    def _centered_binomial_distribution(
        random_buffer: bytes,
        noise_parameter: int,
    ) -> list:
        """Sample small noise coefficients from a centered binomial distribution."""
        random_bits = utils.bytes_to_bits(random_buffer)
        noise_coefficients = []

        for coefficient_index in range(POLYNOMIAL_DEGREE):
            coefficient_bit_start = 2 * coefficient_index * noise_parameter
            positive_bit_sum = sum(
                random_bits[coefficient_bit_start + bit_offset]
                for bit_offset in range(noise_parameter)
            )
            negative_bit_sum = sum(
                random_bits[
                    coefficient_bit_start + noise_parameter + bit_offset
                ]
                for bit_offset in range(noise_parameter)
            )
            noise_coefficients.append(
                (positive_bit_sum - negative_bit_sum) % MODULUS
            )

        return noise_coefficients


    def _sample_noise_polynomial(
        pseudorandom_seed: bytes,
        nonce_value: int,
        noise_parameter: int,
    ) -> list:
        """Expand seed and nonce into one small noise polynomial."""
        random_buffer = utils.shake_256(
            pseudorandom_seed + bytes([nonce_value]),
            64 * noise_parameter,
        )
        return _centered_binomial_distribution(random_buffer, noise_parameter)


    def _generate_public_matrix(matrix_seed: bytes) -> list:
        """Deterministically generate the public MODULE_RANK x MODULE_RANK matrix."""
        public_matrix = [
            [None] * MODULE_RANK
            for _ in range(MODULE_RANK)
        ]

        for row_index in range(MODULE_RANK):
            for column_index in range(MODULE_RANK):
                public_matrix[row_index][column_index] = _parse_uniform(
                    matrix_seed + bytes([row_index, column_index])
                )

        return public_matrix


    # ---------------------------------------------------------------------------
    # Compression, decompression, and message mapping.
    # ---------------------------------------------------------------------------

    def _compress_coefficient(coefficient_value: int, compression_bits: int) -> int:
        return (
            (((coefficient_value % MODULUS) << compression_bits) + (MODULUS // 2))
            // MODULUS
        ) % (1 << compression_bits)


    def _decompress_coefficient(
        compressed_value: int,
        compression_bits: int,
    ) -> int:
        return (
            MODULUS * compressed_value + (1 << (compression_bits - 1))
        ) >> compression_bits


    def _compress_polynomial(polynomial, compression_bits):
        return [
            _compress_coefficient(coefficient, compression_bits)
            for coefficient in polynomial
        ]


    def _decompress_polynomial(polynomial, compression_bits):
        return [
            _decompress_coefficient(coefficient, compression_bits)
            for coefficient in polynomial
        ]


    def _message_to_polynomial(message_bytes: bytes) -> list:
        """Map each bit of a 32-byte message to zero or approximately MODULUS / 2."""
        message_bits = utils.bytes_to_bits(message_bytes)
        return [
            _decompress_coefficient(message_bit, 1)
            for message_bit in message_bits
        ]


    def _polynomial_to_message(polynomial) -> bytes:
        """Recover message bits by choosing the nearest encoded bit value."""
        recovered_bits = [
            _compress_coefficient(coefficient, 1)
            for coefficient in polynomial
        ]
        return utils.bits_to_bytes(recovered_bits)


    # ---------------------------------------------------------------------------
    # Deterministic two-byte-per-coefficient serialization.
    # ---------------------------------------------------------------------------

    def _encode_polynomial(polynomial):
        return b"".join(
            int(coefficient % 65536).to_bytes(2, "big")
            for coefficient in polynomial
        )


    def _decode_polynomial(encoded_polynomial):
        return [
            int.from_bytes(
                encoded_polynomial[byte_index:byte_index + 2],
                "big",
            )
            for byte_index in range(0, len(encoded_polynomial), 2)
        ]


    def _encode_polynomial_vector(polynomial_vector):
        return b"".join(
            _encode_polynomial(polynomial)
            for polynomial in polynomial_vector
        )


    def _decode_polynomial_vector(encoded_vector, vector_length):
        encoded_polynomial_length = POLYNOMIAL_DEGREE * 2
        return [
            _decode_polynomial(
                encoded_vector[
                    vector_index * encoded_polynomial_length:
                    (vector_index + 1) * encoded_polynomial_length
                ]
            )
            for vector_index in range(vector_length)
        ]


    # ---------------------------------------------------------------------------
    # K-PKE, the public-key encryption component used inside ML-KEM.
    # ---------------------------------------------------------------------------

    def _kpke_keygen(key_generation_seed: bytes):
        """Return the encoded public key and the secret polynomial vector."""
        expanded_seed = utils.sha3_512(key_generation_seed)
        matrix_seed = expanded_seed[:32]
        noise_seed = expanded_seed[32:]

        public_matrix = _generate_public_matrix(matrix_seed)

        nonce_counter = 0
        secret_vector = [
            _sample_noise_polynomial(
                noise_seed,
                nonce_counter + vector_index,
                KEY_GENERATION_NOISE_PARAMETER,
            )
            for vector_index in range(MODULE_RANK)
        ]
        nonce_counter += MODULE_RANK

        error_vector = [
            _sample_noise_polynomial(
                noise_seed,
                nonce_counter + vector_index,
                KEY_GENERATION_NOISE_PARAMETER,
            )
            for vector_index in range(MODULE_RANK)
        ]

        public_vector = []
        for row_index in range(MODULE_RANK):
            accumulated_polynomial = [0] * POLYNOMIAL_DEGREE
            for column_index in range(MODULE_RANK):
                accumulated_polynomial = poly_add(
                    accumulated_polynomial,
                    poly_mul(
                        public_matrix[row_index][column_index],
                        secret_vector[column_index],
                    ),
                )
            public_vector.append(
                poly_add(accumulated_polynomial, error_vector[row_index])
            )

        public_key_bytes = _encode_polynomial_vector(public_vector) + matrix_seed
        return public_key_bytes, secret_vector


    def _kpke_encrypt(
        public_key_bytes: bytes,
        message_bytes: bytes,
        encryption_randomness: bytes,
    ) -> bytes:
        """Encrypt a 32-byte message deterministically from encryption_randomness."""
        public_vector = _decode_polynomial_vector(
            public_key_bytes[:-32],
            MODULE_RANK,
        )
        matrix_seed = public_key_bytes[-32:]
        public_matrix = _generate_public_matrix(matrix_seed)

        nonce_counter = 0
        random_vector = [
            _sample_noise_polynomial(
                encryption_randomness,
                nonce_counter + vector_index,
                KEY_GENERATION_NOISE_PARAMETER,
            )
            for vector_index in range(MODULE_RANK)
        ]
        nonce_counter += MODULE_RANK

        vector_error = [
            _sample_noise_polynomial(
                encryption_randomness,
                nonce_counter + vector_index,
                ENCRYPTION_NOISE_PARAMETER,
            )
            for vector_index in range(MODULE_RANK)
        ]
        nonce_counter += MODULE_RANK

        scalar_error = _sample_noise_polynomial(
            encryption_randomness,
            nonce_counter,
            ENCRYPTION_NOISE_PARAMETER,
        )

        ciphertext_vector = []
        for column_index in range(MODULE_RANK):
            accumulated_polynomial = [0] * POLYNOMIAL_DEGREE
            for row_index in range(MODULE_RANK):
                accumulated_polynomial = poly_add(
                    accumulated_polynomial,
                    poly_mul(
                        public_matrix[row_index][column_index],
                        random_vector[row_index],
                    ),
                )
            ciphertext_vector.append(
                poly_add(accumulated_polynomial, vector_error[column_index])
            )

        ciphertext_scalar = [0] * POLYNOMIAL_DEGREE
        for vector_index in range(MODULE_RANK):
            ciphertext_scalar = poly_add(
                ciphertext_scalar,
                poly_mul(public_vector[vector_index], random_vector[vector_index]),
            )
        ciphertext_scalar = poly_add(ciphertext_scalar, scalar_error)
        ciphertext_scalar = poly_add(
            ciphertext_scalar,
            _message_to_polynomial(message_bytes),
        )

        encoded_vector_part = b"".join(
            _encode_polynomial(
                _compress_polynomial(polynomial, VECTOR_COMPRESSION_BITS)
            )
            for polynomial in ciphertext_vector
        )
        encoded_scalar_part = _encode_polynomial(
            _compress_polynomial(ciphertext_scalar, SCALAR_COMPRESSION_BITS)
        )

        return encoded_vector_part + encoded_scalar_part


    def _kpke_decrypt(secret_vector, ciphertext_bytes: bytes) -> bytes:
        encoded_vector_length = MODULE_RANK * POLYNOMIAL_DEGREE * 2
        encoded_vector_part = ciphertext_bytes[:encoded_vector_length]
        encoded_scalar_part = ciphertext_bytes[encoded_vector_length:]

        ciphertext_vector = [
            _decompress_polynomial(
                _decode_polynomial(
                    encoded_vector_part[
                        vector_index * POLYNOMIAL_DEGREE * 2:
                        (vector_index + 1) * POLYNOMIAL_DEGREE * 2
                    ]
                ),
                VECTOR_COMPRESSION_BITS,
            )
            for vector_index in range(MODULE_RANK)
        ]
        ciphertext_scalar = _decompress_polynomial(
            _decode_polynomial(encoded_scalar_part),
            SCALAR_COMPRESSION_BITS,
        )

        secret_vector_product = [0] * POLYNOMIAL_DEGREE
        for vector_index in range(MODULE_RANK):
            secret_vector_product = poly_add(
                secret_vector_product,
                poly_mul(
                    secret_vector[vector_index],
                    ciphertext_vector[vector_index],
                ),
            )

        recovered_message_polynomial = poly_sub(
            ciphertext_scalar,
            secret_vector_product,
        )
        return _polynomial_to_message(recovered_message_polynomial)


    # ---------------------------------------------------------------------------
    # ML-KEM interface with the Fujisaki-Okamoto transform.
    # ---------------------------------------------------------------------------

    def keygen():
        """Generate and return (public_key, private_key)."""
        key_generation_seed = os.urandom(32)
        implicit_rejection_secret = os.urandom(32)
        public_key, secret_vector = _kpke_keygen(key_generation_seed)

        private_key = {
            "secret_vector": secret_vector,
            "public_key": public_key,
            "public_key_hash": utils.sha3_256(public_key),
            "implicit_rejection_secret": implicit_rejection_secret,
        }
        return public_key, private_key


    def encapsulate(public_key: bytes):
        """Create and return (shared_secret, kyber_ciphertext)."""
        random_message = os.urandom(32)
        hash_output = utils.sha3_512(
            random_message + utils.sha3_256(public_key)
        )
        shared_secret = hash_output[:32]
        encryption_randomness = hash_output[32:]
        kyber_ciphertext = _kpke_encrypt(
            public_key,
            random_message,
            encryption_randomness,
        )
        return shared_secret, kyber_ciphertext


    def decapsulate(private_key: dict, kyber_ciphertext: bytes) -> bytes:
        """Recover the shared secret, or derive a replacement after tampering."""
        recovered_message = _kpke_decrypt(
            private_key["secret_vector"],
            kyber_ciphertext,
        )
        hash_output = utils.sha3_512(
            recovered_message + private_key["public_key_hash"]
        )
        candidate_shared_secret = hash_output[:32]
        candidate_encryption_randomness = hash_output[32:]

        recomputed_ciphertext = _kpke_encrypt(
            private_key["public_key"],
            recovered_message,
            candidate_encryption_randomness,
        )

        if recomputed_ciphertext == kyber_ciphertext:
            return candidate_shared_secret

        return utils.shake_256(
            private_key["implicit_rejection_secret"] + kyber_ciphertext,
            32,
        )


    return SimpleNamespace(
        keygen=keygen,
        encapsulate=encapsulate,
        decapsulate=decapsulate,
        poly_add=poly_add,
        poly_sub=poly_sub,
        poly_mul=poly_mul,
        POLYNOMIAL_DEGREE=POLYNOMIAL_DEGREE,
        MODULUS=MODULUS,
        MODULE_RANK=MODULE_RANK
    )

kyber = build_kyber_module(utils)



In [ ]:
# @title Cell 5 — Falcon-Style Digital Signature
from types import SimpleNamespace

def build_falcon_signature_module(utility_module):
    utils = utility_module
    """
    falcon_ntru.py
    --------------
    An educational, from-scratch Falcon-style digital signature implementation.

    The public verification relation is:

        signature_first_polynomial
        + signature_second_polynomial * public_polynomial
        == Hash(message)  (mod modulus)

    A valid signature must also be short according to the public squared-norm
    bound. This project uses a very small ring, LLL basis reduction, and Babai's
    nearest-plane algorithm so that the mechanism remains readable and executable.
    It demonstrates the structure of Falcon/NTRU signatures but is not intended
    for production security.
    """

    import os


    POLYNOMIAL_DEGREE = 16
    MODULUS = 12289


    # ---------------------------------------------------------------------------
    # Polynomial arithmetic in Z_q[x] / (x^POLYNOMIAL_DEGREE + 1).
    # ---------------------------------------------------------------------------

    def _multiply_polynomials_mod_q(first_polynomial, second_polynomial):
        """Multiply two polynomials modulo x^n + 1 and MODULUS."""
        unreduced_product = [0] * (2 * POLYNOMIAL_DEGREE)

        for first_degree in range(POLYNOMIAL_DEGREE):
            for second_degree in range(POLYNOMIAL_DEGREE):
                unreduced_product[first_degree + second_degree] += (
                    first_polynomial[first_degree]
                    * second_polynomial[second_degree]
                )

        return [
            (
                unreduced_product[coefficient_index]
                - unreduced_product[
                    coefficient_index + POLYNOMIAL_DEGREE
                ]
            ) % MODULUS
            for coefficient_index in range(POLYNOMIAL_DEGREE)
        ]


    def _build_negacyclic_matrix(polynomial):
        """Build the matrix representing multiplication by polynomial in the ring."""
        negacyclic_matrix = [
            [0] * POLYNOMIAL_DEGREE
            for _ in range(POLYNOMIAL_DEGREE)
        ]

        for row_index in range(POLYNOMIAL_DEGREE):
            for column_index in range(POLYNOMIAL_DEGREE):
                coefficient_offset = row_index - column_index
                if coefficient_offset >= 0:
                    negacyclic_matrix[row_index][column_index] = (
                        polynomial[coefficient_offset]
                    )
                else:
                    negacyclic_matrix[row_index][column_index] = -polynomial[
                        coefficient_offset + POLYNOMIAL_DEGREE
                    ]

        return negacyclic_matrix


    def _solve_linear_system_mod_q(coefficient_matrix, result_vector):
        """Solve coefficient_matrix * solution = result_vector modulo MODULUS."""
        matrix_size = len(coefficient_matrix)
        augmented_matrix = [
            [
                coefficient_matrix[row_index][column_index] % MODULUS
                for column_index in range(matrix_size)
            ]
            + [result_vector[row_index] % MODULUS]
            for row_index in range(matrix_size)
        ]

        for pivot_column in range(matrix_size):
            pivot_row = next(
                (
                    candidate_row
                    for candidate_row in range(pivot_column, matrix_size)
                    if augmented_matrix[candidate_row][pivot_column] % MODULUS != 0
                ),
                None,
            )
            if pivot_row is None:
                return None

            augmented_matrix[pivot_column], augmented_matrix[pivot_row] = (
                augmented_matrix[pivot_row],
                augmented_matrix[pivot_column],
            )

            pivot_inverse = pow(
                augmented_matrix[pivot_column][pivot_column],
                -1,
                MODULUS,
            )
            augmented_matrix[pivot_column] = [
                (entry * pivot_inverse) % MODULUS
                for entry in augmented_matrix[pivot_column]
            ]

            for row_index in range(matrix_size):
                if (
                    row_index != pivot_column
                    and augmented_matrix[row_index][pivot_column] != 0
                ):
                    elimination_factor = augmented_matrix[row_index][pivot_column]
                    augmented_matrix[row_index] = [
                        (
                            augmented_matrix[row_index][entry_index]
                            - elimination_factor
                            * augmented_matrix[pivot_column][entry_index]
                        ) % MODULUS
                        for entry_index in range(matrix_size + 1)
                    ]

        return [
            augmented_matrix[row_index][matrix_size] % MODULUS
            for row_index in range(matrix_size)
        ]


    def _center_coefficients(polynomial):
        """Map coefficients to the symmetric interval around zero."""
        return [
            ((coefficient + MODULUS // 2) % MODULUS) - MODULUS // 2
            for coefficient in polynomial
        ]


    # ---------------------------------------------------------------------------
    # Lattice helpers: dot product, Gram-Schmidt, LLL, and Babai.
    # ---------------------------------------------------------------------------

    def _dot_product(first_vector, second_vector):
        return sum(
            first_value * second_value
            for first_value, second_value
            in zip(first_vector, second_vector)
        )


    def _compute_gram_schmidt(lattice_basis):
        """Return the orthogonal basis and Gram-Schmidt coefficients."""
        basis_size = len(lattice_basis)
        orthogonal_basis = [
            [float(coordinate) for coordinate in basis_vector]
            for basis_vector in lattice_basis
        ]
        projection_coefficients = [
            [0.0] * basis_size
            for _ in range(basis_size)
        ]

        for basis_index in range(basis_size):
            for previous_basis_index in range(basis_index):
                denominator = _dot_product(
                    orthogonal_basis[previous_basis_index],
                    orthogonal_basis[previous_basis_index],
                )
                if denominator:
                    projection_coefficients[basis_index][previous_basis_index] = (
                        _dot_product(
                            lattice_basis[basis_index],
                            orthogonal_basis[previous_basis_index],
                        )
                        / denominator
                    )
                else:
                    projection_coefficients[basis_index][previous_basis_index] = 0.0

                projection_value = projection_coefficients[
                    basis_index
                ][previous_basis_index]
                orthogonal_basis[basis_index] = [
                    orthogonal_basis[basis_index][coordinate_index]
                    - projection_value
                    * orthogonal_basis[previous_basis_index][coordinate_index]
                    for coordinate_index in range(
                        len(lattice_basis[basis_index])
                    )
                ]

        return orthogonal_basis, projection_coefficients


    def _reduce_lattice_basis_lll(lattice_basis, reduction_parameter=0.99):
        """Reduce an integer lattice basis using the LLL algorithm."""
        reduced_basis = [list(basis_vector) for basis_vector in lattice_basis]
        basis_size = len(reduced_basis)
        orthogonal_basis, projection_coefficients = _compute_gram_schmidt(
            reduced_basis
        )
        orthogonal_squared_norms = [
            _dot_product(
                orthogonal_basis[basis_index],
                orthogonal_basis[basis_index],
            )
            for basis_index in range(basis_size)
        ]

        current_basis_index = 1
        iteration_count = 0

        while current_basis_index < basis_size and iteration_count < 200000:
            iteration_count += 1

            for previous_basis_index in range(
                current_basis_index - 1,
                -1,
                -1,
            ):
                current_projection = projection_coefficients[
                    current_basis_index
                ][previous_basis_index]

                if abs(current_projection) > 0.5:
                    reduction_multiplier = round(current_projection)
                    reduced_basis[current_basis_index] = [
                        reduced_basis[current_basis_index][coordinate_index]
                        - reduction_multiplier
                        * reduced_basis[previous_basis_index][coordinate_index]
                        for coordinate_index in range(
                            len(reduced_basis[current_basis_index])
                        )
                    ]

                    for earlier_basis_index in range(previous_basis_index):
                        projection_coefficients[
                            current_basis_index
                        ][earlier_basis_index] -= (
                            reduction_multiplier
                            * projection_coefficients[
                                previous_basis_index
                            ][earlier_basis_index]
                        )

                    projection_coefficients[
                        current_basis_index
                    ][previous_basis_index] -= reduction_multiplier

            lovasz_right_side = (
                reduction_parameter
                - projection_coefficients[
                    current_basis_index
                ][current_basis_index - 1] ** 2
            ) * orthogonal_squared_norms[current_basis_index - 1]

            if orthogonal_squared_norms[current_basis_index] >= lovasz_right_side:
                current_basis_index += 1
            else:
                reduced_basis[current_basis_index], reduced_basis[
                    current_basis_index - 1
                ] = (
                    reduced_basis[current_basis_index - 1],
                    reduced_basis[current_basis_index],
                )

                orthogonal_basis, projection_coefficients = _compute_gram_schmidt(
                    reduced_basis
                )
                orthogonal_squared_norms = [
                    _dot_product(
                        orthogonal_basis[basis_index],
                        orthogonal_basis[basis_index],
                    )
                    for basis_index in range(basis_size)
                ]
                current_basis_index = max(current_basis_index - 1, 1)

        return reduced_basis


    def _babai_nearest_plane(
        reduced_basis,
        orthogonal_basis,
        target_vector,
    ):
        """Return target minus Babai's nearest lattice vector approximation."""
        residual_vector = list(target_vector)

        for basis_index in range(len(reduced_basis) - 1, -1, -1):
            nearest_integer_coefficient = round(
                _dot_product(
                    residual_vector,
                    orthogonal_basis[basis_index],
                )
                / _dot_product(
                    orthogonal_basis[basis_index],
                    orthogonal_basis[basis_index],
                )
            )
            residual_vector = [
                residual_vector[coordinate_index]
                - nearest_integer_coefficient
                * reduced_basis[basis_index][coordinate_index]
                for coordinate_index in range(len(residual_vector))
            ]

        return residual_vector


    # ---------------------------------------------------------------------------
    # Hash a message into one polynomial in Z_q^n.
    # ---------------------------------------------------------------------------

    def _hash_to_point(message_bytes: bytes) -> list:
        hash_stream = utils.shake_256(
            b"FALCON-EDU" + message_bytes,
            2 * POLYNOMIAL_DEGREE,
        )
        return [
            int.from_bytes(
                hash_stream[
                    2 * coefficient_index:2 * coefficient_index + 2
                ],
                "big",
            ) % MODULUS
            for coefficient_index in range(POLYNOMIAL_DEGREE)
        ]


    def _generate_small_polynomial():
        """Generate coefficients independently from {-1, 0, 1}."""
        return [
            int.from_bytes(os.urandom(1), "big") % 3 - 1
            for _ in range(POLYNOMIAL_DEGREE)
        ]


    # ---------------------------------------------------------------------------
    # Key generation.
    # ---------------------------------------------------------------------------

    def keygen():
        """Generate and return (public_key, private_key)."""
        while True:
            private_polynomial_f = _generate_small_polynomial()
            private_polynomial_g = _generate_small_polynomial()

            if all(
                coefficient == 0
                for coefficient in private_polynomial_f
            ):
                continue

            multiplication_matrix_f = _build_negacyclic_matrix(
                private_polynomial_f
            )
            multiplicative_identity_polynomial = [1] + [
                0
            ] * (POLYNOMIAL_DEGREE - 1)
            inverse_polynomial_f = _solve_linear_system_mod_q(
                multiplication_matrix_f,
                multiplicative_identity_polynomial,
            )

            if inverse_polynomial_f is None:
                continue

            public_polynomial = _multiply_polynomials_mod_q(
                private_polynomial_g,
                inverse_polynomial_f,
            )
            break

        centered_public_polynomial = _center_coefficients(public_polynomial)
        negative_public_matrix = _build_negacyclic_matrix(
            [-coefficient for coefficient in centered_public_polynomial]
        )

        lattice_dimension = 2 * POLYNOMIAL_DEGREE
        public_lattice_basis = [
            [0] * lattice_dimension
            for _ in range(lattice_dimension)
        ]

        for polynomial_index in range(POLYNOMIAL_DEGREE):
            public_lattice_basis[polynomial_index][polynomial_index] = MODULUS

        for polynomial_index in range(POLYNOMIAL_DEGREE):
            for coefficient_index in range(POLYNOMIAL_DEGREE):
                public_lattice_basis[
                    POLYNOMIAL_DEGREE + polynomial_index
                ][coefficient_index] = negative_public_matrix[
                    coefficient_index
                ][polynomial_index]

            public_lattice_basis[
                POLYNOMIAL_DEGREE + polynomial_index
            ][POLYNOMIAL_DEGREE + polynomial_index] = 1

        reduced_lattice_basis = _reduce_lattice_basis_lll(public_lattice_basis)
        orthogonal_lattice_basis, _ = _compute_gram_schmidt(
            reduced_lattice_basis
        )

        private_key = {
            "reduced_lattice_basis": reduced_lattice_basis,
            "orthogonal_lattice_basis": orthogonal_lattice_basis,
        }

        maximum_squared_norm = 0
        for _ in range(8):
            first_signature_polynomial, second_signature_polynomial = (
                _sign_raw(private_key, os.urandom(16))
            )
            signature_squared_norm = (
                _dot_product(
                    first_signature_polynomial,
                    first_signature_polynomial,
                )
                + _dot_product(
                    second_signature_polynomial,
                    second_signature_polynomial,
                )
            )
            maximum_squared_norm = max(
                maximum_squared_norm,
                signature_squared_norm,
            )

        squared_norm_bound = int(maximum_squared_norm * 1.7) + 1
        public_key = {
            "public_polynomial": public_polynomial,
            "squared_norm_bound": squared_norm_bound,
        }

        return public_key, private_key


    # ---------------------------------------------------------------------------
    # Signing and verification.
    # ---------------------------------------------------------------------------

    def _sign_raw(private_key, message_bytes: bytes):
        """Produce short polynomials satisfying the Falcon verification relation."""
        challenge_polynomial = _hash_to_point(message_bytes)
        target_vector = challenge_polynomial + [0] * POLYNOMIAL_DEGREE
        short_residual_vector = _babai_nearest_plane(
            private_key["reduced_lattice_basis"],
            private_key["orthogonal_lattice_basis"],
            target_vector,
        )
        return (
            short_residual_vector[:POLYNOMIAL_DEGREE],
            short_residual_vector[POLYNOMIAL_DEGREE:],
        )


    def sign(private_key, message_bytes: bytes) -> bytes:
        """Sign message_bytes and serialize the second signature polynomial."""
        _, second_signature_polynomial = _sign_raw(private_key, message_bytes)
        return b"".join(
            int(coefficient % 65536).to_bytes(2, "big")
            for coefficient in second_signature_polynomial
        )


    def verify(public_key, message_bytes: bytes, signature_bytes: bytes) -> bool:
        """Verify the public relation and the squared-norm bound."""
        if len(signature_bytes) != 2 * POLYNOMIAL_DEGREE:
            return False

        second_signature_polynomial = [
            int.from_bytes(
                signature_bytes[
                    2 * coefficient_index:2 * coefficient_index + 2
                ],
                "big",
            )
            for coefficient_index in range(POLYNOMIAL_DEGREE)
        ]
        second_signature_polynomial = [
            coefficient - 65536 if coefficient >= 32768 else coefficient
            for coefficient in second_signature_polynomial
        ]

        challenge_polynomial = _hash_to_point(message_bytes)
        signature_times_public_key = _multiply_polynomials_mod_q(
            [
                coefficient % MODULUS
                for coefficient in second_signature_polynomial
            ],
            public_key["public_polynomial"],
        )
        first_signature_polynomial = _center_coefficients(
            [
                (
                    challenge_polynomial[coefficient_index]
                    - signature_times_public_key[coefficient_index]
                ) % MODULUS
                for coefficient_index in range(POLYNOMIAL_DEGREE)
            ]
        )

        signature_squared_norm = (
            _dot_product(
                first_signature_polynomial,
                first_signature_polynomial,
            )
            + _dot_product(
                second_signature_polynomial,
                second_signature_polynomial,
            )
        )
        return signature_squared_norm <= public_key["squared_norm_bound"]


    return SimpleNamespace(
        keygen=keygen,
        sign=sign,
        verify=verify,
        POLYNOMIAL_DEGREE=POLYNOMIAL_DEGREE,
        MODULUS=MODULUS
    )

falcon_signature = build_falcon_signature_module(utils)


In [ ]:
# @title Cell 6 — Complete Secure Exchange Flow
"""
=========================================================================
SECURE MEDICAL RECORD EXCHANGE
=========================================================================

A patient sends a medical record to a doctor over an insecure channel.
The project combines three cryptographic components:

1. ML-KEM / CRYSTALS-Kyber
   The patient encapsulates a shared secret with the doctor's public key.
   Only the doctor's private key can recover the same shared secret.

2. IDEA in CFB mode
   A 128-bit IDEA key is derived from the shared secret and is used to
   encrypt the medical record.

3. Falcon-style NTRU signature
   The patient signs the complete transmitted package. The doctor verifies
   the signature before recovering the key or decrypting the record.

The implementation is educational and is not intended for production use.
=========================================================================
"""


# ---------------------------------------------------------------------------
# Package serialization.
# Each part is stored as: [4-byte length][part bytes].
# This gives the doctor an unambiguous way to split the received package.
# ---------------------------------------------------------------------------

def _pack_byte_parts(*byte_parts: bytes) -> bytes:
    """Serialize multiple byte strings into one length-prefixed package."""
    serialized_package = b""
    for current_part in byte_parts:
        part_length = len(current_part)
        serialized_package += part_length.to_bytes(4, "big") + current_part
    return serialized_package


def _unpack_byte_parts(serialized_package: bytes) -> list:
    """Recover the byte strings produced by _pack_byte_parts."""
    recovered_parts = []
    current_byte_index = 0

    while current_byte_index < len(serialized_package):
        if current_byte_index + 4 > len(serialized_package):
            raise ValueError("Invalid package: missing length field")

        part_length = int.from_bytes(
            serialized_package[current_byte_index:current_byte_index + 4],
            "big",
        )
        current_byte_index += 4

        part_end_index = current_byte_index + part_length
        if part_end_index > len(serialized_package):
            raise ValueError("Invalid package: declared part length is too large")

        recovered_parts.append(serialized_package[current_byte_index:part_end_index])
        current_byte_index = part_end_index

    return recovered_parts


# ---------------------------------------------------------------------------
# Complete project flow.
# All cryptographic operations run first. All output is printed together at
# the end of this function in the same order as the protocol stages.
# ---------------------------------------------------------------------------

def main():

    # The medical record was defined in Cell 1 and is stored in memory.
    medical_record_source = MEDICAL_RECORD_SOURCE_DESCRIPTION
    original_medical_record = MEDICAL_RECORD_BYTES

    # Step 1: The doctor creates the ML-KEM key pair used for key delivery.
    doctor_kyber_public_key, doctor_kyber_private_key = kyber.keygen()

    # Step 1: The patient creates the Falcon key pair used for signing.
    patient_falcon_public_key, patient_falcon_private_key = falcon_signature.keygen()

    # Step 2: The patient creates a shared secret for the doctor.
    patient_shared_secret, kyber_encapsulation_ciphertext = kyber.encapsulate(
        doctor_kyber_public_key
    )

    # Step 2: Derive the 128-bit IDEA key from the 256-bit shared secret.
    patient_idea_encryption_key = utils.sha3_256(
        b"IDEA-KEY" + patient_shared_secret
    )[:16]

    # Step 3: Encrypt the medical record using IDEA in CFB mode.
    idea_initialization_vector = idea_cfb.random_iv()
    encrypted_medical_record = idea_cfb.encrypt(
        patient_idea_encryption_key,
        idea_initialization_vector,
        original_medical_record,
    )

    # Step 4: Build one package and sign every transmitted byte.
    signed_package = _pack_byte_parts(
        kyber_encapsulation_ciphertext,
        idea_initialization_vector,
        encrypted_medical_record,
    )
    patient_falcon_signature = falcon_signature.sign(
        patient_falcon_private_key,
        signed_package,
    )

    transmitted_bundle = {
        "signed_package": signed_package,
        "patient_signature": patient_falcon_signature,
    }

    # Step 5: The doctor verifies authenticity and integrity first.
    received_signature_is_valid = falcon_signature.verify(
        patient_falcon_public_key,
        transmitted_bundle["signed_package"],
        transmitted_bundle["patient_signature"],
    )

    doctor_recovered_shared_secret = None
    doctor_idea_decryption_key = None
    recovered_medical_record = None
    recovered_secret_matches = False
    recovered_record_matches = False
    tampered_package_signature_is_valid = None
    tampered_kyber_package_signature_is_valid = None
    tampered_kyber_secret_is_different = None

    if received_signature_is_valid:
        (
            doctor_received_kyber_ciphertext,
            doctor_received_initialization_vector,
            doctor_received_encrypted_record,
        ) = _unpack_byte_parts(transmitted_bundle["signed_package"])

        # Step 6: The doctor decapsulates the shared secret.
        doctor_recovered_shared_secret = kyber.decapsulate(
            doctor_kyber_private_key,
            doctor_received_kyber_ciphertext,
        )
        recovered_secret_matches = doctor_recovered_shared_secret == patient_shared_secret

        doctor_idea_decryption_key = utils.sha3_256(
            b"IDEA-KEY" + doctor_recovered_shared_secret
        )[:16]

        # Step 7: The doctor decrypts the medical record.
        recovered_medical_record = idea_cfb.decrypt(
            doctor_idea_decryption_key,
            doctor_received_initialization_vector,
            doctor_received_encrypted_record,
        )
        recovered_record_matches = recovered_medical_record == original_medical_record

        # Step 8A: Change one byte anywhere inside the signed package.
        tampered_package_buffer = bytearray(transmitted_bundle["signed_package"])
        tampered_package_buffer[-1] ^= 0x01
        tampered_package = bytes(tampered_package_buffer)

        tampered_package_signature_is_valid = falcon_signature.verify(
            patient_falcon_public_key,
            tampered_package,
            transmitted_bundle["patient_signature"],
        )

        # Step 8B: Change the Kyber ciphertext inside the signed package.
        # In the real receiver flow, the doctor checks the signature first,
        # so this modified package must be rejected before decapsulation.
        tampered_kyber_ciphertext_buffer = bytearray(doctor_received_kyber_ciphertext)
        tampered_kyber_ciphertext_buffer[0] ^= 0x01
        tampered_kyber_ciphertext = bytes(tampered_kyber_ciphertext_buffer)

        package_with_tampered_kyber_ciphertext = _pack_byte_parts(
            tampered_kyber_ciphertext,
            doctor_received_initialization_vector,
            doctor_received_encrypted_record,
        )

        tampered_kyber_package_signature_is_valid = falcon_signature.verify(
            patient_falcon_public_key,
            package_with_tampered_kyber_ciphertext,
            transmitted_bundle["patient_signature"],
        )

        # This extra educational check shows the ML-KEM implicit rejection
        # behavior: even if someone tried to decapsulate a modified Kyber
        # ciphertext directly, it would not recover the original secret.
        secret_from_tampered_kyber_ciphertext = kyber.decapsulate(
            doctor_kyber_private_key,
            tampered_kyber_ciphertext,
        )
        tampered_kyber_secret_is_different = (
            secret_from_tampered_kyber_ciphertext != patient_shared_secret
        )
    # -----------------------------------------------------------------------
    # Ordered project report. These are the only print operations in the
    # complete project, and they all remain inside main().
    # -----------------------------------------------------------------------
    separator_line = "-" * 78
    strong_separator_line = "=" * 78

    print(strong_separator_line)
    print("               SECURE MEDICAL RECORD EXCHANGE")
    print("       IDEA-CFB | ML-KEM (Kyber) | Falcon-style Signature")
    print(strong_separator_line)

    print("\n[0] MEDICAL RECORD INPUT")
    print("    The patient loads the medical record that will be protected.")
    print(f"    Input source                      : {medical_record_source}")
    print(f"    Original record size              : {len(original_medical_record)} bytes")

    print(f"\n{separator_line}")
    print("[1] PUBLIC-KEY AND SIGNATURE KEY GENERATION")
    print("    Doctor: creates an ML-KEM key pair for receiving the shared secret.")
    print("    Patient: creates a Falcon key pair for signing the transmitted data.")
    print(f"    Doctor ML-KEM public key size      : {len(doctor_kyber_public_key)} bytes")
    print("    Doctor ML-KEM private key          : stored privately by the doctor")
    print("    Patient Falcon public key          : available to the doctor")
    print("    Patient Falcon private key         : stored privately by the patient")

    print(f"\n{separator_line}")
    print("[2] SHARED-SECRET DELIVERY WITH ML-KEM")
    print("    The patient encapsulates a fresh shared secret using the doctor's")
    print("    public key. The resulting Kyber ciphertext can travel openly.")
    print(f"    Kyber ciphertext size              : {len(kyber_encapsulation_ciphertext)} bytes")
    print(f"    Shared secret size                 : {len(patient_shared_secret)} bytes")
    print("    Shared secret value                : hidden")
    print("    IDEA key derivation                : SHA3-256('IDEA-KEY' || secret)[:16]")
    print(f"    Derived IDEA key size              : {len(patient_idea_encryption_key)} bytes")

    print(f"\n{separator_line}")
    print("[3] MEDICAL RECORD ENCRYPTION WITH IDEA-CFB")
    print("    The patient encrypts the complete record with the derived IDEA key.")
    print("    CFB mode requires no padding and preserves the original byte length.")
    print(f"    Initialization vector size         : {len(idea_initialization_vector)} bytes")
    print(f"    Encrypted record size              : {len(encrypted_medical_record)} bytes")
    print(f"    First 32 encrypted bytes           : {encrypted_medical_record[:32].hex()}")
    print(f"    Ciphertext SHA3-256 fingerprint    : {utils.sha3_256(encrypted_medical_record).hex()}")
    print(f"\n{separator_line}")
    print("[4] PACKAGE CREATION AND FALCON SIGNING")
    print("    The patient combines the Kyber ciphertext, IDEA initialization vector,")
    print("    and encrypted medical record into one length-prefixed package.")
    print("    The Falcon signature covers the complete package before transmission.")
    print(f"    Signed package size                : {len(signed_package)} bytes")
    print(f"    Falcon signature size              : {len(patient_falcon_signature)} bytes")
    print("    Transmitted bundle                 : signed_package + patient_signature")

    print(f"\n{strong_separator_line}")
    print("                       DOCTOR RECEIVER FLOW")
    print(strong_separator_line)

    print("\n[5] FALCON SIGNATURE VERIFICATION")
    print("    The doctor verifies the patient's signature before processing any data.")
    print(f"    Signature is valid                 : {received_signature_is_valid}")

    if not received_signature_is_valid:
        print("    Result                             : package rejected; processing stopped")
        print(f"\n{strong_separator_line}")
        print("PROJECT RESULT: FAILED AT SIGNATURE VERIFICATION")
        print(strong_separator_line)
        return

    print("    Result                             : authenticity and integrity confirmed")

    print(f"\n{separator_line}")
    print("[6] SHARED-SECRET RECOVERY WITH ML-KEM")
    print("    The doctor decapsulates the received Kyber ciphertext with the private key.")
    print(f"    Recovered secret matches patient   : {recovered_secret_matches}")
    print(f"    Recovered IDEA key size            : {len(doctor_idea_decryption_key)} bytes")

    print(f"\n{separator_line}")
    print("[7] MEDICAL RECORD DECRYPTION WITH IDEA-CFB")
    print("    The doctor decrypts the ciphertext with the recovered IDEA key and IV.")
    print(f"    Decrypted record matches original  : {recovered_record_matches}")
    print("\n    Recovered medical record - first 12 lines:")
    for recovered_record_line in recovered_medical_record.decode(
        "utf-8",
        "replace",
    ).splitlines()[:12]:
        print(f"    | {recovered_record_line}")

    print(f"\n{separator_line}")
    print("[8] INTEGRITY AND TAMPERING VALIDATION")
    print("    Test A: an attacker changes one byte in the signed encrypted package.")
    print(f"    Modified package signature valid   : {tampered_package_signature_is_valid}")
    print("    Doctor action                      : reject because expected value is False")

    print("\n    Test B: an attacker changes one byte in the Kyber ciphertext part.")
    print(f"    Modified Kyber package signature   : {tampered_kyber_package_signature_is_valid}")
    print("    Doctor action                      : reject before decapsulation/decryption")
    print(f"    Extra ML-KEM check: secret changed : {tampered_kyber_secret_is_different}")

    print(f"\n{strong_separator_line}")
    print("PROJECT RESULT: ALL REQUIRED SECURITY FLOWS COMPLETED SUCCESSFULLY")
    print("Confidentiality: IDEA-CFB encrypted the medical record.")
    print("Integrity: Falcon rejected modified transmitted data.")
    print("Authenticity: the patient's Falcon public key verified the signature.")
    print("Key delivery: ML-KEM allowed only the doctor's private key to recover the secret.")
    print(strong_separator_line)


In [ ]:
# @title Cell 7 — Run Complete Project and Print Results
main()

               SECURE MEDICAL RECORD EXCHANGE
       IDEA-CFB | ML-KEM (Kyber) | Falcon-style Signature

[0] MEDICAL RECORD INPUT
    The patient loads the medical record that will be protected.
    Input source                      : Colab memory variable: MEDICAL_RECORD_TEXT (no external file)
    Original record size              : 1803 bytes

------------------------------------------------------------------------------
[1] PUBLIC-KEY AND SIGNATURE KEY GENERATION
    Doctor: creates an ML-KEM key pair for receiving the shared secret.
    Patient: creates a Falcon key pair for signing the transmitted data.
    Doctor ML-KEM public key size      : 1056 bytes
    Doctor ML-KEM private key          : stored privately by the doctor
    Patient Falcon public key          : available to the doctor
    Patient Falcon private key         : stored privately by the patient

------------------------------------------------------------------------------
[2] SHARED-SECRET DELIVERY WITH ML-KEM
  